# Time Dimension Data - Gold Layer

## Objective
Extract and deduplicate unique time attributes from silver.silver_multimodal to build a standardized Time Dimension Gold Delta table (gold.gold_multimodal_dim_time).

## Data Flow
silver.silver_multimodal → Spark SQL / DataFrame → gold.gold_multimodal_dim_time

## Source
The underlying data comes from the multimodal transport network API.

## Input
Silver Delta table: silver.silver_multimodal

## Output
Gold Delta table: gold.gold_multimodal_dim_time

## Gold Layer Principle
The Gold layer delivers curated, dimensional models and business-level aggregations ready for reporting and analytics. This pipeline isolates unique intraday temporal attributes and categorizes time periods to maintain a clean dimension table with strict entity integrity.

## Processing Steps
1. **Load Silver Data:** Read silver.silver_multimodal into PySpark.
2. **Extract Dimension:** Query distinct timestamp values and calculate surrogate time keys, hour/minute components, and time period categories.
3. **Write to Gold:** Persist deduplicated dataset to gold.gold_multimodal_dim_time Delta table.

In [0]:
# Load data from silver schema
df_silver_multimodal=spark.table('workspace.silver.silver_multimodal')

In [0]:
# display the dataframe 
df_silver_multimodal.display()

## BUSINESS TRANSFORMATION AND MODELING

In [0]:
# Extract  Dimension time Data
query_dim_time = """
SELECT DISTINCT
    CAST(hour(CAST(timestamp AS TIMESTAMP)) * 10000 + minute(CAST(timestamp AS TIMESTAMP)) * 100 AS INT) AS time_key,
    hour(CAST(timestamp AS TIMESTAMP)) AS hour,
    minute(CAST(timestamp AS TIMESTAMP)) AS minute,
    date_format(CAST(timestamp AS TIMESTAMP), 'HH:mm') AS hour_label,
    CASE 
        WHEN hour(CAST(timestamp AS TIMESTAMP)) BETWEEN 7 AND 9 THEN 'Morning'
        WHEN hour(CAST(timestamp AS TIMESTAMP)) BETWEEN 17 AND 19 THEN 'Evening'
        WHEN hour(CAST(timestamp AS TIMESTAMP)) BETWEEN 10 AND 16 THEN 'Daytime'
        ELSE 'Night'
    END AS time_period
FROM workspace.silver.silver_multimodal
"""

df_dim_time = spark.sql(query_dim_time)

In [0]:
# Display df_dim_time 
df_dim_time.display()

# WRITING GOLD TABLE

In [0]:
df_dim_time\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("gold.gold_multimodal_dim_time")

# CHECKING THE GOLD TABLE

In [0]:
%sql
SELECT * 
FROM workspace.gold.gold_multimodal_dim_time
LIMIT 10